# Reproduction of the diagnostics and figures of the manuscript

This is an attempt to be a step-by-step reproduction of the figures and
tables of the manuscript in order to :

1.  check that the `Downclim` package is able to reproduce the results
    already obtained
2.  have a fully reproducible workflow

The workflow here is far from being optimal for typical `Downclim`
usage. This is more intended for a detailed and explicit workflow, with
explicit function calls.

In [ ]:
from downclim import aoi, downscale, evaluation
from downclim.dataset import gshtd, chirps, chelsa2, cmip6, cordex
import ee
import pandas as pd

In [ ]:
ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com", project="downclim")

## Define the areas of interest

In [ ]:
nc = aoi.get_aoi("New-Caledonia", save_aoi_file=True)
ic = aoi.get_aoi("Côte d'Ivoire", save_aoi_file=True)
guy = aoi.get_aoi("Guyana", save_aoi_file=True)

## Variable and periods definition

In [ ]:
variables = ["tas", "tasmin", "tasmax", "pr"]
historical_period = (1980, 2005)
evaluation_period = (2006, 2019)
projection_period = (2071, 2100)

## Baseline product

We’re now downloading the baseline dataset `Chelsa2`.

In [ ]:
chelsa2.get_chelsa2(
    aoi = [nc, ic, guy],
    variable = variables,
    period = historical_period,
    keep_tmp_dir = True,
    )

## Evaluation products

We’re now downloading the evaluation datasets `Chirps` and `GSHTD`. They are available en google earth engine.

In [ ]:
ee.Initialize(opt_url="https://earthengine-highvolume.googleapis.com", project="downclim")

chirps.get_chirps(
    aoi = [nc, ic, guy],
    variable = variables,
    period = evaluation_period,
    )

gshtd.get_gshtd(
    aoi = [nc, ic, guy],
    variable = variables,
    period = evaluation_period,
    )

## Getting global simulations

We’re now getting simulations. First global `CMIP6` simulations.

In [ ]:
cmip6context = cmip6.CMIP6Context(
  experiment = ["historical", "ssp126", "ssp585"],
  variable = variables,
)
cmip6_simulations = cmip6context.list_available_simulations(save_simulations="./cmip6_simulations.csv")

And finally we get the simulations:

In [ ]:
cmip6.get_cmip6(
    [nc, ic, guy],
    cmip6_simulations= cmip6_simulations,
    historical_period = historical_period,
    evaluation_period = evaluation_period,
    projection_period = projection_period,
    )

## Getting regional simulations

Then we will search for available simulations:

In [ ]:
cordexcontext = cordex.CORDEXContext(
  experiment = ["historical", "rcp26", "rcp85"],
  domain = ["AUS-22", "SAM-22", "AFR-22"],
  variable = variables,
)

cordex_simulations = cordexcontext.list_available_simulations(
    save_simulations="./cordex_simulations.csv",
    server="https://esg-dn1.nsc.liu.se/esg-search",
)

And finally we get the simulations:

In [ ]:
cordex.get_cordex(
    [nc, ic, guy],
    cordex_simulations= cordex_simulations,
    historical_period = historical_period,
    evaluation_period = evaluation_period,
    projection_period = projection_period,
    nb_threads = 4,
)

## Downscale the simulations

We now have a set of `CMIP6` and `CORDEX` simulations, we want to
downscale them. We will downscale all simulations using `CHELSA` as the
baseline product, and using the `CHELSA` grid.

In [ ]:
downscale.run_downscaling(
    aoi=[nc, ic, guy],
    historical_period=historical_period,
    evaluation_period=evaluation_period,
    projection_period=projection_period,
    baseline_product=utils.DataProduct.CHELSA,
    cmip6_simulations_to_downscale=None,   # use all files in ./results
    cordex_simulations_to_downscale=None,  # use all files in ./results
    downscaling_grid_file=None,            # use baseline grid
)

## Evaluate the downscaled simulations

Finally we evaluate the downscaled simulations. Evaluation products are
`CHIRPS` and `GSHTD`. We evaluate the downscaled simulations along these
products on the respective CHELSA grid for all aois.

In [ ]:
evaluation.run_evaluation(
    aoi=[nc, ic, guy],
    evaluation_period=evaluation_period,
    evaluation_product=[utils.DataProduct.CHIRPS, utils.DataProduct.GSHTD],
    evaluation_grid=utils.DataProduct.CHELSA,
)